# Tool로 AgentCore Memory(장기)를 사용하는 Strands Agents

## 개요
이 Notebook에서는 Strands와 AgentCore Memory를 사용하여 대화형 AI agent에 장기 메모리 기능을 구현하는 방법을 살펴봅니다. 단기 상호 작용에서 중요한 정보를 추출하고 통합하여 agent가 시간이 지나고 여러 대화 session이 이어져도 핵심 세부 정보를 기억하도록 구성하는 방법을 학습합니다.

## 튜토리얼 세부 정보
**사용 사례:** 지속형 메모리를 사용하는 요리 도우미

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                                       |
| Agent 유형          | 요리 도우미                                                                       |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 'User Preferences' Memory Extraction, 메모리 저장 및 검색용 Memory Tool |
| 예제 난이도         | 초급                                                                              |

다음 내용을 학습합니다.
- 장기 보존을 위한 extraction strategy로 AgentCore Memory 구성
- 이전 대화 기록으로 메모리 채우기
- 장기 메모리를 사용하여 여러 대화 session에 걸쳐 개인화된 경험 제공
- Strands Agent Framework와 AgentCore Memory tool 통합

## 시나리오 배경

이 튜토리얼에서는 고도로 개인화된 음식점 추천을 제공하는 요리 도우미 역할을 수행합니다. AgentCore Memory의 장기 보존 및 자동 정보 추출 기능을 활용하면 agent가 식단 선택이나 좋아하는 요리 같은 사용자 선호도를 여러 대화에 걸쳐 기억할 수 있습니다. 이 지속형 메모리를 통해 대화가 며칠 또는 몇 주간 이어져도 맞춤형 제안과 끊김 없는 사용자 경험을 제공할 수 있습니다. 이 시나리오는 구조화된 메모리 구성과 설정 가능한 strategy를 통해 대화형 AI가 단기 기억을 넘어 흥미롭고 맥락을 이해하는 상호 작용을 구현하는 방법을 보여 줍니다.


## 아키텍처

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
- Python 3.10+
- Amazon Bedrock AgentCore Memory 권한이 있는 AWS credentials
- Amazon Bedrock AgentCore SDK
환경을 설정하고 적절한 extraction strategy가 적용된 장기 메모리 resource를 생성하며 시작해 보겠습니다!

## 1단계: 환경 설정
이 Notebook 실행에 필요한 모든 library를 import하고 client를 정의하겠습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import time
import logging
from datetime import datetime

Amazon Bedrock model 및 AgentCore에 필요한 권한이 있는 region과 role을 정의합니다.

In [ ]:
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("culinary-memory")

region = os.getenv("AWS_REGION", "us-west-2")

## 2단계: 장기 Strategy로 Memory 생성

이 섹션에서는 장기 메모리 기능으로 구성된 memory resource를 생성합니다. 이전의 단기 메모리 예제와 달리 이 구현에는 정보를 통합하여 보존할 수 있는 특정 memory strategy가 포함됩니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=region)

memory_name = "CulinaryAssistant"
memory_id = None

In [ ]:
from botocore.exceptions import ClientError

try:
    print("Creating Long-Term Memory...")

    # 장기 메모리 resource에 더 구체적인 이름 사용
    memory_name = memory_name

    # User preference strategy로 memory 생성
    memory = client.create_memory_and_wait(
        name=memory_name,
        description="Culinary Assistant Agent with long term memory",
        strategies=[
            {
                StrategyType.USER_PREFERENCE.value: {
                    "name": "UserPreferences",
                    "description": "Captures user preferences",
                    "namespaceTemplates": ["/user/{actorId}/preferences/"],
                }
            }
        ],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10,
    )

    memory_id = memory["id"]
    print(f"Memory created successfully with ID: {memory_id}")

except ClientError as e:
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        # Memory가 이미 있으면 ID 검색
        memories = client.list_memories()
        memory_id = next((m["id"] for m in memories if m["id"].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Memory 생성 중 발생하는 오류 처리
    logger.info(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()
    # 오류 발생 시 정리 - memory가 일부 생성되었다면 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

### 장기 Memory Strategy 이해

이 memory 생성 과정의 핵심 차이점은 **memory strategy**가 추가된다는 것입니다. 구성 요소를 살펴보겠습니다.

#### 1. User Preference Memory Strategy

이 strategy는 대화에서 사용자 선호도를 자동으로 식별하고 추출합니다.

```python
"userPreferenceMemoryStrategy": {
    "name": "UserPreferences",
    "description": "Captures user preferences",
    "namespaceTemplates": ["/user/{actorId}/preferences/"]
}
```

#### 2. Memory Namespace

`namespaceTemplates` parameter는 추출된 정보의 저장 위치를 정의합니다.

```python
"namespaceTemplates": ["/user/{actorId}/preferences/"]
```

이 memory strategy는 단순히 대화를 기억하는 데 그치지 않고, 대화 속 중요한 정보를 실제로 이해하고 구성하여 나중에 활용할 수 있는 정교한 memory system을 만듭니다.

## 3단계: 이전 대화를 Memory에 저장

이 섹션에서는 단기 메모리를 채워 background의 장기 메모리 추출 process를 자동으로 시작하는 방법을 살펴봅니다.

### 단기 메모리 채우기

Extraction strategy로 구성된 memory resource에 대화를 저장하면 추가 코드 없이도 system이 장기 보존을 위해 이 정보를 자동으로 처리합니다.

In [ ]:
actor_id = f"user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"foodie-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"/user/{actor_id}/preferences/"

In [ ]:
previous_messages = [
    ("Hi, I'm John", "USER"),
    ("Hi John, how can I help you with food recommendations today?", "ASSISTANT"),
    ("I'm looking for some vegetarian dishes to try this weekend.", "USER"),
    (
        "That sounds great! I'd be happy to help with vegetarian recommendations. Do you have any specific ingredients or cuisine types you prefer?",
        "ASSISTANT",
    ),
    ("Yes, I really like tofu and fresh vegetables in my dishes", "USER"),
    (
        "Perfect! Tofu and fresh vegetables make for excellent vegetarian meals. I can suggest some stir-fries, Buddha bowls, or tofu curries. Do you have any other preferences?",
        "ASSISTANT",
    ),
    (
        "I also really enjoy Italian cuisine. I love pasta dishes and would like them to be vegetarian-friendly.",
        "USER",
    ),
    (
        "Excellent! Italian cuisine has wonderful vegetarian options. I can recommend pasta primavera, mushroom risotto, eggplant parmesan, or penne arrabbiata. The combination of Italian flavors with vegetarian ingredients creates delicious meals!",
        "ASSISTANT",
    ),
    (
        "I spent 2 hours looking through cookbooks but couldn't find inspiring vegetarian Italian recipes",
        "USER",
    ),
    (
        "I'm sorry you had trouble finding inspiring recipes! Let me help you with some creative vegetarian Italian dishes. How about stuffed bell peppers with Italian herbs and rice, spinach and ricotta cannelloni, or a Mediterranean vegetable lasagna?",
        "ASSISTANT",
    ),
    ("Hey, I appreciate food assistants with good taste", "USER"),
    (
        "Ha! I definitely try to bring good taste to the table! Speaking of which, shall we explore some more vegetarian Italian recipes that might inspire you?",
        "ASSISTANT",
    ),
]

In [ ]:
print("\nHydrating short term memory with previous conversations...")

# 대화 기록을 단기 메모리에 저장
initial = client.create_event(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    messages=previous_messages,
)
print("✓ Conversation saved in short term memory")

대화 message가 포함된 event가 올바르게 저장되었는지 확인해 보겠습니다.

In [ ]:
events = client.list_events(memory_id=memory_id, actor_id=actor_id, session_id=session_id, max_results=5)
events

이 셀은 실행 중 유용한 message를 표시하도록 logging system을 구성하여 코드 실행 상황을 추적할 수 있게 합니다.

### Background에서 일어나는 작업

`create_event` 호출 후 다음 작업이 자동으로 수행됩니다.

1. **단기 저장**: 전체 대화를 원본 형식으로 저장
2. **추출 시작**: Memory system이 이 memory에 UserPreference strategy가 구성된 것을 감지
3. **Background 처리**: 추가 코드 없이 system이 다음 작업 수행
   - 대화에서 선호도를 나타내는 표현 분석
   - "I'm vegetarian" 및 "I really enjoy Italian cuisine" 같은 문장 식별
   - 해당 선호도를 구조화된 data로 추출
4. **장기 통합**: 추출된 선호도를 구성된 namespace(`/user/{actorId}/preferences/`)에 저장

추출과 통합은 자동으로 수행됩니다. Agent와 대화를 이어 가거나 단기 메모리를 채우기만 하면 memory 생성 시 구성한 strategy가 나머지를 처리합니다.

이 자동 process를 통해 단기 대화 record가 만료된 후에도 중요한 정보가 장기 메모리에 보존됩니다.


## 장기 메모리 검색

이 섹션에서는 장기 메모리에 저장된 추출된 선호도에 액세스하는 방법을 살펴봅니다. 대화 turn에 초점을 맞춘 단기 메모리 검색과 달리, 장기 메모리 검색은 추출 및 통합된 구조화된 정보에 액세스하는 데 중점을 둡니다.

### 장기 메모리의 User Preferences에 액세스

장기 메모리에서 정보를 검색하려면 memory 생성 시 정의한 namespace 구조를 사용합니다.


In [ ]:
# Memory 추출에서 event를 처리할 시간을 확보하도록 30초 대기 추가
time.sleep(30)

try:
    # 음식 선호도를 memory system에 질의
    food_preferences = client.retrieve_memories(
        memory_id=memory_id,
        namespace=namespace,
        query="food preferences",
        top_k=3,  # 관련성이 가장 높은 결과를 최대 3개 반환
    )

    if food_preferences:
        print(f"Retrieved {len(food_preferences)} relevant preference records:")
        for i, record in enumerate(food_preferences):
            print(f"\nMemory {i + 1}:")
            print(f"- Content: {record.get('content', 'Not specified')}")
    else:
        print("No matching preference records found.")

except Exception as e:
    print(f"Error retrieving preference records: {e}")

이 method를 사용하면 필요할 때 관련 메모리를 검색할 수 있습니다. 기본 사항을 익혔으니 이제 agent를 만들어 보겠습니다!

## 4단계: Agent 생성
이 섹션에서는 기본 제공 `agent_core_memory` tool을 사용하여 AgentCore Memory를 Strands Agent와 통합하는 방법을 살펴봅니다.

#### 장기 메모리 기능으로 Agent 설정
메모리 지원 agent를 생성하기 위해 Strands framework를 사용하고 AgentCore Memory resource에 연결합니다.

In [ ]:
from strands import Agent
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

In [ ]:
system_prompt = """You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.

PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide personalized dining recommendations

You have access to a Memory tool that enables you to:
- Store user preferences (dietary restrictions, favorite cuisines, budget preferences, etc.)
- Retrieve previously stored information to personalize recommendations

"""

In [ ]:
provider = AgentCoreMemoryToolProvider(
    memory_id=memory_id, actor_id=actor_id, session_id=session_id, namespace=namespace
)

agent = Agent(
    tools=provider.tools,
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt=system_prompt,
)

이미 단기 및 장기 메모리를 채웠으므로 agent에서 메모리를 직접 검색해 보겠습니다!

In [ ]:
agent("Give me restaurant recommendations in Irvine based on my food preferences")

Agent는 retrieve_memory_records method를 사용하여 사용자의 메모리를 검색해야 합니다.

좋습니다! 이제 AgentCore 장기 메모리에서 메모리를 검색할 수 있는 Strands Agent가 완성되었습니다!

## 정리
이 Notebook에서 사용한 resource를 정리하도록 memory를 삭제하겠습니다.

In [ ]:
# client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
# )